In [595]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import re

In [596]:
dia = "20260421"

fecha = '2026-04-21'

Perdidos actual

In [597]:
# perdidos = pd.read_csv(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/Descargas dia actual FMS/perdidos/{dia}_bitacora perdidos.csv', encoding='latin')

# perdidos.head()

perdidos vencidos

In [598]:
perdidos = pd.read_csv(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/Datos diarios dia vencido FMS/perdidos/{dia}_bitacora perdidos.csv', encoding='latin')

perdidos.head()

,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,PARADA FIN,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO
0,2026-04-21,BC29D0012,SE14,OSCAR ORLANDO DAZA GUERRA,506373.0,1.122404e+09,04:41:15,06:41:30,Gestion del Operador,Operador en otro servicio,...,554A12_TM,"36,341",0,"36,341",1.0,MIRIAM CRISTINA RAMIREZ CUERVO,2026-04-21 04:01:30,NO,NaN,PATIO TINTAL 2
1,2026-04-21,BC29D0012,SE14,OSCAR ORLANDO DAZA GUERRA,506373.0,1.122404e+09,06:41:30,09:29:30,Gestion del Operador,Operador en otro servicio,...,072A05_TM_(Engativa),"36,909",0,"36,909",1.0,JULIAN ANDRES SANCHEZ RESTREPO,2026-04-21 04:58:52,NO,NaN,PATIO TINTAL 2
2,2026-04-21,BC29D0018,SE14,JOSE ALBERTO CARDENAS OSPINO,507427.0,1.065995e+09,10:07:00,12:49:00,Gestion del Operador,Operador en otro servicio,...,554A12_TM,"36,341",0,"36,341",2.0,WILLIAM IGNACIO VALDERRAMA GONZALEZ,2026-04-21 10:38:05,NO,Z50-7049,PATIO TINTAL 2
3,2026-04-21,BC29D0018,SE14,JONATHAN MUÑOZ MOLANO,504928.0,1.022341e+09,15:40:30,18:40:30,Gestion de SV,Accidente,...,554A12_Br. Diana Turbay Cultivos,"36,341","12,504","23,837",1.0,JOHN JAIRO ESPINOSA SANTA,2026-04-21 17:19:36,NO,Z50-7077,PATIO TINTAL 2
4,2026-04-21,BC29D0018,SE14,JONATHAN MUÑOZ MOLANO,504928.0,1.022341e+09,18:41:15,21:16:15,Gestion de Mtto,Movil no despachado,...,072A05_TM_(Engativa),"36,909",0,"36,909",1.0,JOHN JAIRO ESPINOSA SANTA,2026-04-21 17:19:55,NO,Z50-7077,PATIO TINTAL 2


Datos

In [599]:
#Rutas asociadas
base = pd.read_excel(
    'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/rutas.xlsx',
    sheet_name='rutas'
)

#Motivos de eliminación
motivos = pd.read_excel(
    'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Indicadores/Base_indicadores/motivos_eliminacion.xlsx',
    sheet_name='rutas'
)

base.head()

,Id Línea,Nombre Línea,Ruta,Nombre Ruta,Sublínea
0,10014,M86-K86,10176,M86 - K86,115
1,10011,L81-D81,10022,L81_Ref_CL72,20
2,10011,L81-D81,10025,D81_Normal,21
3,10011,L81-D81,10026,L81_Normal,21
4,10010,H83-M83,10023,H83,22


IPH

In [600]:
#IPH GM

iph = pd.read_excel(
    'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/iph_general.xlsx',
    sheet_name='Iph_general'
)

iph['Validacion'] = 1

iph.head()

,Fecha,Jornada,TipoDia,Concesionario de Operación,Instante,Servicio Bus,Evento,Id Línea,Tabla,Sublinea,Id Ruta,Id Nodo,Tipo Nodo,Viaje,Servicio Conductor entrante,Turno Entrante,Operador Entrante,Servicio Conductor Saliente,Tipo Vehiculo,Validacion
0,2025-12-01,GC251201T2,CN0010014301,GMOVIL ENGATIVA,4:30:00,CN08F0010,18,10339,7,NaN,NaN,142,Patio,1,CE130935,1.0,105.0,NaN,BUS (80),1
1,2025-12-01,GC251201T2,CN0010014301,GMOVIL ENGATIVA,5:10:00,CN08F0010,4,10339,7,NaN,NaN,52372,Parada,1,NaN,NaN,NaN,NaN,BUS (80),1
2,2025-12-01,GC251201T2,CN0010014301,GMOVIL ENGATIVA,5:10:00,CN08F0010,11,10339,7,2348.0,12606.0,52372,Parada,2,NaN,NaN,NaN,NaN,BUS (80),1
3,2025-12-01,GC251201T2,CN0010014301,GMOVIL ENGATIVA,5:31:27,CN08F0010,0,10339,7,2348.0,12606.0,53222,Parada,2,NaN,NaN,NaN,NaN,BUS (80),1
4,2025-12-01,GC251201T2,CN0010014301,GMOVIL ENGATIVA,5:42:53,CN08F0010,0,10339,7,2348.0,12606.0,52686,Parada,2,NaN,NaN,NaN,NaN,BUS (80),1


In [601]:
# columna Fecha en formato fecha
iph['Fecha'] = pd.to_datetime(iph['Fecha'], errors='coerce')

# Filtrar solo esa fecha
iph= iph[iph['Fecha'] == fecha].copy()

#reemplazar faltantes
iph = iph.replace(['nan','NaN','NAN',''], 0)
iph = iph.fillna(0)

#Convertir a entero
iph['Sublinea'] = iph['Sublinea'].astype(int)
iph['Id Ruta'] = iph['Id Ruta'].astype(int)

# Ver resultado
iph.head()

,Fecha,Jornada,TipoDia,Concesionario de Operación,Instante,Servicio Bus,Evento,Id Línea,Tabla,Sublinea,Id Ruta,Id Nodo,Tipo Nodo,Viaje,Servicio Conductor entrante,Turno Entrante,Operador Entrante,Servicio Conductor Saliente,Tipo Vehiculo,Validacion


Datos brutos

In [602]:
acciones_fms = pd.read_excel(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/datos_brutos_FMS/{dia}_datos_brutos.xlsx')

acciones_fms.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,REACT_DATETIME,MOTIVE_ID,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,TGT_UNDO_SEQ
0,20260421,556306,105,10273,1.0,NaN,20,com.lgcns.fms.oprt.schedule.driver.model.Chang...,CE1390001,N,NaN,0,WMUNAR_105,20260421005515,NaN,NaN,NaN
1,20260421,556307,105,10273,1.0,NaN,20,com.lgcns.fms.oprt.schedule.driver.model.Chang...,CE1390001,N,NaN,0,WMUNAR_105,20260421005559,NaN,NaN,NaN
2,20260421,556308,105,10273,9.0,NaN,20,com.lgcns.fms.oprt.schedule.driver.model.Chang...,CE1390009,N,NaN,0,WMUNAR_105,20260421005626,NaN,NaN,NaN
3,20260421,556312,105,10325,16.0,NaN,20,com.lgcns.fms.oprt.schedule.driver.model.Chang...,CE1320016,N,NaN,0,WMUNAR_105,20260421005901,NaN,NaN,NaN
4,20260421,556315,105,10292,4.0,NaN,20,com.lgcns.fms.oprt.schedule.driver.model.Chang...,CE1780010,N,NaN,0,WMUNAR_105,20260421010044,NaN,NaN,NaN


Cambiar coche

In [603]:
#Accion de regulacion de 
acciones_change_vehicle = acciones_fms.copy()

acciones_change_vehicle = acciones_change_vehicle[acciones_change_vehicle['REGUL_TYPE_ID'] == 36].copy()
acciones_change_vehicle['LINE_SERV_ID'] = acciones_change_vehicle['LINE_SERV_ID'].astype(int)
acciones_change_vehicle['VEH_REGISTR_NUM'] = acciones_change_vehicle['VEH_REGISTR_NUM'].astype(int)

acciones_change_vehicle.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,REACT_DATETIME,MOTIVE_ID,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,TGT_UNDO_SEQ
610,20260421,560028,105,10550,7,504031,36,<= Change Vehicle =>\nIdLinea=10550\nServBusRe...,CE05D0007,N,NaN,26,SVALERIANO_105,20260421065115,NaN,NaN,NaN
622,20260421,560131,105,10194,13,507043,36,<= Change Vehicle =>\nIdLinea=10194\nServBusRe...,CE1690011,N,NaN,26,LGONZALEZ_105,20260421065647,NaN,NaN,NaN
628,20260421,560196,105,10690,7,504297,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210007,N,NaN,26,YHERNANDEZ_105,20260421070042,NaN,NaN,NaN
641,20260421,560311,105,10690,8,504132,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210008,N,NaN,26,YHERNANDEZ_105,20260421070806,NaN,NaN,NaN
642,20260421,560313,105,10261,8,507063,36,<= Change Vehicle =>\nIdLinea=10261\nServBusRe...,CE13F0008,N,NaN,26,WVALDERRAMA_105,20260421070812,NaN,NaN,NaN


In [604]:
#Función para convertir columnas
def param_to_dict(texto):
    d = {}
    
    if pd.isna(texto):
        return d
    
    # convertir a string y separar por saltos de línea
    lineas = str(texto).split('\n')
    
    for linea in lineas:
        linea = linea.strip()
        
        if '=' in linea:
            try:
                clave, valor = linea.split('=', 1)
                
                # limpiar [] si existen
                clave = clave.replace('[', '').replace(']', '').strip()
                valor = valor.strip()
                
                # convertir 'null' a NaN
                if valor.lower() in ['null', 'nan', 'none', '']:
                    valor = None
                
                d[clave] = valor
            except:
                pass

    return d

In [605]:
# Convierte PARAM_VALUE en diccionario
param_df = acciones_change_vehicle['PARAM_VALUE'].apply(param_to_dict)

# Convertir a DataFrame y unir con el original
param_df = pd.DataFrame(param_df.tolist())

acciones_change_vehicle = pd.concat([acciones_change_vehicle.reset_index(drop=True), param_df.reset_index(drop=True)], axis=1)

acciones_change_vehicle.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,IdConductorRef,IdRutaDesdeRef,ViajeLineaDesdeRef,IdViajeDesdeRef,IdRutaHastaRef,ViajeLineaHastaRef,IdViajeHastaRef,ServBusNuevo,IdConductorNuevo,SimulationYn
0,20260421,560028,105,10550,7,504031,36,<= Change Vehicle =>\nIdLinea=10550\nServBusRe...,CE05D0007,N,...,510354,11081,2,3,11081,2,3,CE05DG007,502440,N
1,20260421,560131,105,10194,13,507043,36,<= Change Vehicle =>\nIdLinea=10194\nServBusRe...,CE1690011,N,...,508710,12785,2,3,12785,2,3,CE169G011,506999,N
2,20260421,560196,105,10690,7,504297,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210007,N,...,510316,12721,2,3,12721,2,3,CE121G007,509019,N
3,20260421,560311,105,10690,8,504132,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210008,N,...,510483,12721,2,3,12721,2,3,CE121G008,509276,N
4,20260421,560313,105,10261,8,507063,36,<= Change Vehicle =>\nIdLinea=10261\nServBusRe...,CE13F0008,N,...,507497,12759,2,3,12759,2,3,CE13FG008,508836,N


In [606]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (base['Id Línea '] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not base.loc[filtro].empty:
        # Obtener el primer valor
        tipo = base.loc[filtro, 'Nombre Línea '].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_change_vehicle['Nombre Línea'] = acciones_change_vehicle.apply(
    lambda row: calcular_posicion(
        row['LINE_ID']
    ),
    axis=1
)

acciones_change_vehicle['Nombre Línea'] = acciones_change_vehicle['Nombre Línea'].fillna(0)

acciones_change_vehicle.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,IdRutaDesdeRef,ViajeLineaDesdeRef,IdViajeDesdeRef,IdRutaHastaRef,ViajeLineaHastaRef,IdViajeHastaRef,ServBusNuevo,IdConductorNuevo,SimulationYn,Nombre Línea
0,20260421,560028,105,10550,7,504031,36,<= Change Vehicle =>\nIdLinea=10550\nServBusRe...,CE05D0007,N,...,11081,2,3,11081,2,3,CE05DG007,502440,N,P500
1,20260421,560131,105,10194,13,507043,36,<= Change Vehicle =>\nIdLinea=10194\nServBusRe...,CE1690011,N,...,12785,2,3,12785,2,3,CE169G011,506999,N,539
2,20260421,560196,105,10690,7,504297,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210007,N,...,12721,2,3,12721,2,3,CE121G007,509019,N,DL219
3,20260421,560311,105,10690,8,504132,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210008,N,...,12721,2,3,12721,2,3,CE121G008,509276,N,DL219
4,20260421,560313,105,10261,8,507063,36,<= Change Vehicle =>\nIdLinea=10261\nServBusRe...,CE13F0008,N,...,12759,2,3,12759,2,3,CE13FG008,508836,N,466


In [607]:
acciones_change_vehicle = acciones_change_vehicle[acciones_change_vehicle['CREAT_USER_ID'].astype(str).str.endswith('_105', na=False)].copy()

acciones_change_vehicle.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,IdRutaDesdeRef,ViajeLineaDesdeRef,IdViajeDesdeRef,IdRutaHastaRef,ViajeLineaHastaRef,IdViajeHastaRef,ServBusNuevo,IdConductorNuevo,SimulationYn,Nombre Línea
0,20260421,560028,105,10550,7,504031,36,<= Change Vehicle =>\nIdLinea=10550\nServBusRe...,CE05D0007,N,...,11081,2,3,11081,2,3,CE05DG007,502440,N,P500
1,20260421,560131,105,10194,13,507043,36,<= Change Vehicle =>\nIdLinea=10194\nServBusRe...,CE1690011,N,...,12785,2,3,12785,2,3,CE169G011,506999,N,539
2,20260421,560196,105,10690,7,504297,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210007,N,...,12721,2,3,12721,2,3,CE121G007,509019,N,DL219
3,20260421,560311,105,10690,8,504132,36,<= Change Vehicle =>\nIdLinea=10690\nServBusRe...,CE1210008,N,...,12721,2,3,12721,2,3,CE121G008,509276,N,DL219
4,20260421,560313,105,10261,8,507063,36,<= Change Vehicle =>\nIdLinea=10261\nServBusRe...,CE13F0008,N,...,12759,2,3,12759,2,3,CE13FG008,508836,N,466


Introducir coche

In [608]:
#Accion de regulacion de CreateServiceVo
CreateServiceVo = acciones_fms.copy()

CreateServiceVo = CreateServiceVo[CreateServiceVo['REGUL_TYPE_ID'] == 4].copy()
CreateServiceVo['LINE_SERV_ID'] = CreateServiceVo['LINE_SERV_ID'].astype(int)
CreateServiceVo['VEH_REGISTR_NUM'] = CreateServiceVo['VEH_REGISTR_NUM'].astype(int)

CreateServiceVo.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,REACT_DATETIME,MOTIVE_ID,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,TGT_UNDO_SEQ
293,20260421,558053,0,10065,139,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0065139,N,NaN,39,SHERNANDEZ_TM,20260421045658,NaN,NaN,NaN
327,20260421,558237,0,10054,79,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0054079,N,NaN,39,OMONROY_TM,20260421050831,NaN,NaN,NaN
351,20260421,558435,0,10072,50,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0072050,N,NaN,39,JAMONTES_TM,20260421051915,NaN,NaN,NaN
395,20260421,558578,0,10072,51,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0072051,N,NaN,39,JAMONTES_TM,20260421052706,NaN,NaN,NaN
396,20260421,558580,0,10591,102,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0591102,N,NaN,39,LDIAZ_TM,20260421052711,NaN,NaN,NaN


In [609]:
#Función para convertir columnas
def param_to_dict(texto):
    d = {}
    
    if pd.isna(texto):
        return d
    
    # convertir a string y separar por saltos de línea
    lineas = str(texto).split('\n')
    
    for linea in lineas:
        linea = linea.strip()
        
        if '=' in linea:
            try:
                clave, valor = linea.split('=', 1)
                
                # limpiar [] si existen
                clave = clave.replace('[', '').replace(']', '').strip()
                valor = valor.strip()
                
                # convertir 'null' a NaN
                if valor.lower() in ['null', 'nan', 'none', '']:
                    valor = None
                
                d[clave] = valor
            except:
                pass

    return d

In [610]:
# Convierte PARAM_VALUE en diccionario
param_df = CreateServiceVo['PARAM_VALUE'].apply(param_to_dict)

# Convertir a DataFrame y unir con el original
param_df = pd.DataFrame(param_df.tolist())

CreateServiceVo = pd.concat([CreateServiceVo.reset_index(drop=True), param_df.reset_index(drop=True)], axis=1)

CreateServiceVo.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,simulationYn,timeGap,toNodeEventSecond,toNodeId,toNodeTypeCd,toServTripSeq,userId,validStartDatetime,vehRegistrNum,vehServId
0,20260421,558053,0,10065,139,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0065139,N,...,N,-2220,28605,62286,1,3,SHERNANDEZ_TM,<null>,0,TTAA30976
1,20260421,558237,0,10054,79,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0054079,N,...,N,4815,25185,61837,1,3,OMONROY_TM,<null>,0,TTAA30047
2,20260421,558435,0,10072,50,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0072050,N,...,N,4800,19200,61689,1,3,JAMONTES_TM,<null>,0,TTAA30013
3,20260421,558578,0,10072,51,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0072051,N,...,N,4710,29565,74179,1,6,JAMONTES_TM,<null>,0,TTAA30040
4,20260421,558580,0,10591,102,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0591102,N,...,N,5160,16845,61550,1,2,LDIAZ_TM,<null>,0,TTAA30019


In [611]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (base['Id Línea '] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not base.loc[filtro].empty:
        # Obtener el primer valor
        tipo = base.loc[filtro, 'Nombre Línea '].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
CreateServiceVo['Nombre Línea'] = CreateServiceVo.apply(
    lambda row: calcular_posicion(
        row['LINE_ID']
    ),
    axis=1
)

CreateServiceVo['Nombre Línea'] = CreateServiceVo['Nombre Línea'].fillna(0)

CreateServiceVo.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,timeGap,toNodeEventSecond,toNodeId,toNodeTypeCd,toServTripSeq,userId,validStartDatetime,vehRegistrNum,vehServId,Nombre Línea
0,20260421,558053,0,10065,139,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0065139,N,...,-2220,28605,62286,1,3,SHERNANDEZ_TM,<null>,0,TTAA30976,B46-G46
1,20260421,558237,0,10054,79,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0054079,N,...,4815,25185,61837,1,3,OMONROY_TM,<null>,0,TTAA30047,C17-H17
2,20260421,558435,0,10072,50,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0072050,N,...,4800,19200,61689,1,3,JAMONTES_TM,<null>,0,TTAA30013,5
3,20260421,558578,0,10072,51,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0072051,N,...,4710,29565,74179,1,6,JAMONTES_TM,<null>,0,TTAA30040,5
4,20260421,558580,0,10591,102,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0591102,N,...,5160,16845,61550,1,2,LDIAZ_TM,<null>,0,TTAA30019,E48-G48


In [612]:
if 'userId' in CreateServiceVo.columns:
    CreateServiceVo = CreateServiceVo[
        CreateServiceVo['userId']
        .astype(str)
        .str.endswith('_105', na=False)
    ].copy()
else:
    print("La columna userId no existe, se omite el filtro")

CreateServiceVo.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,timeGap,toNodeEventSecond,toNodeId,toNodeTypeCd,toServTripSeq,userId,validStartDatetime,vehRegistrNum,vehServId,Nombre Línea
11,20260421,559450,105,10474,13,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0474013,N,...,0,33360,73719,1,8,LVARGAS_105,<null>,0,CE0A40005,16-2 Engativa Centro
18,20260421,560745,105,10495,4,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0495004,N,...,0,28500,73713,1,9,LVARGAS_105,<null>,0,CE0A40016,16-14 Aeropuerto
39,20260421,571742,105,10474,14,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0474014,N,...,0,75540,73719,1,15,SCORREA_105,<null>,0,CE0A40017,16-2 Engativa Centro
40,20260421,572232,105,10690,47,0,4,com.lgcns.fms.oprt.regulation.addservice.model...,AD0690047,N,...,0,73170,52372,1,4,JGUZMAN_105,<null>,0,CE1210031,DL219


Acciones de eliminacion

In [613]:
acciones_fms = acciones_fms[acciones_fms['REGUL_TYPE_ID'] == 5].copy()

acciones_fms['LINE_SERV_ID'] = acciones_fms['LINE_SERV_ID'].astype(int)
acciones_fms['VEH_REGISTR_NUM'] = acciones_fms['VEH_REGISTR_NUM'].astype(int)

acciones_fms.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,REACT_DATETIME,MOTIVE_ID,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,TGT_UNDO_SEQ
45,20260421,556516,101,10338,3,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450003,N,NaN,1,LFORERO_101,20260421024341,NaN,NaN,NaN
46,20260421,556520,101,10338,5,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450005,N,NaN,1,LFORERO_101,20260421024357,NaN,NaN,NaN
47,20260421,556526,101,10338,11,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450011,N,NaN,1,LFORERO_101,20260421024443,NaN,NaN,NaN
48,20260421,556533,101,10338,16,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450016,N,NaN,1,LFORERO_101,20260421024518,NaN,NaN,NaN
82,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,NaN,14,KBARRERA_105,20260421032238,NaN,NaN,NaN


In [614]:
#Función para convertir columnas
def param_to_dict(texto):
    d = {}
    
    if pd.isna(texto):
        return d
    
    # convertir a string y separar por saltos de línea
    lineas = str(texto).split('\n')
    
    for linea in lineas:
        linea = linea.strip()
        
        if '=' in linea:
            try:
                clave, valor = linea.split('=', 1)
                
                # limpiar [] si existen
                clave = clave.replace('[', '').replace(']', '').strip()
                valor = valor.strip()
                
                # convertir 'null' a NaN
                if valor.lower() in ['null', 'nan', 'none', '']:
                    valor = None
                
                d[clave] = valor
            except:
                pass

    return d

In [615]:
# Convierte PARAM_VALUE en diccionario
param_df = acciones_fms['PARAM_VALUE'].apply(param_to_dict)

# Convertir a DataFrame y unir con el original
param_df = pd.DataFrame(param_df.tolist())

acciones_fms = pd.concat([acciones_fms.reset_index(drop=True), param_df.reset_index(drop=True)], axis=1)

acciones_fms.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(15),KmEliminado(16),KmEliminado(17),KmEliminado(20),KmEliminado(18),KmEliminado(19),KmEliminado(26),KmEliminado(25),KmEliminado(29),KmEliminado(27)
0,20260421,556516,101,10338,3,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450003,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20260421,556520,101,10338,5,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450005,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20260421,556526,101,10338,11,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450011,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260421,556533,101,10338,16,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450016,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [616]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (base['Id Línea '] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not base.loc[filtro].empty:
        # Obtener el primer valor
        tipo = base.loc[filtro, 'Nombre Línea '].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms['Nombre Línea'] = acciones_fms.apply(
    lambda row: calcular_posicion(
        row['LINE_ID']
    ),
    axis=1
)

acciones_fms['Nombre Línea'] = acciones_fms['Nombre Línea'].fillna(0)

acciones_fms.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(16),KmEliminado(17),KmEliminado(20),KmEliminado(18),KmEliminado(19),KmEliminado(26),KmEliminado(25),KmEliminado(29),KmEliminado(27),Nombre Línea
0,20260421,556516,101,10338,3,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450003,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,20260421,556520,101,10338,5,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450005,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,20260421,556526,101,10338,11,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450011,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,20260421,556533,101,10338,16,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450016,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DL219


In [617]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (motivos['IdMotivoElim'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not motivos.loc[filtro].empty:
        # Obtener el primer valor
        tipo = motivos.loc[filtro, 'Descripción Motivo Elim'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms['Descripción Motivo Elim'] = acciones_fms.apply(
    lambda row: calcular_posicion(
        row['MOTIVE_ID']
    ),
    axis=1
)

acciones_fms.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(17),KmEliminado(20),KmEliminado(18),KmEliminado(19),KmEliminado(26),KmEliminado(25),KmEliminado(29),KmEliminado(27),Nombre Línea,Descripción Motivo Elim
0,20260421,556516,101,10338,3,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450003,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo
1,20260421,556520,101,10338,5,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450005,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo
2,20260421,556526,101,10338,11,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450011,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo
3,20260421,556533,101,10338,16,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450016,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DL219,No se presenta conductor a realizar servicio


In [618]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (iph['Servicio Bus'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not iph.loc[filtro].empty:
        # Obtener el primer valor
        tipo = iph.loc[filtro, 'Validacion'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms['Validacion'] = acciones_fms.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)

acciones_fms['Validacion'] = acciones_fms['Validacion'].fillna(0)
acciones_fms['Validacion'] = acciones_fms['Validacion'].astype(int)

acciones_fms.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\502293850.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  acciones_fms['Validacion'] = acciones_fms['Validacion'].fillna(0)


,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(20),KmEliminado(18),KmEliminado(19),KmEliminado(26),KmEliminado(25),KmEliminado(29),KmEliminado(27),Nombre Línea,Descripción Motivo Elim,Validacion
0,20260421,556516,101,10338,3,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450003,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo,0
1,20260421,556520,101,10338,5,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450005,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo,0
2,20260421,556526,101,10338,11,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450011,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo,0
3,20260421,556533,101,10338,16,0,5,<= Elimination Vehicle =>\nIdLinea=10338\nServ...,CN2450016,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Concesionario no envía vehículo,0
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DL219,No se presenta conductor a realizar servicio,0


In [619]:
acciones_fms_f = acciones_fms.copy()

In [620]:
acciones_fms_f = acciones_fms_f[acciones_fms_f['CREAT_USER_ID'].astype(str).str.endswith('_105', na=False)].copy()

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(20),KmEliminado(18),KmEliminado(19),KmEliminado(26),KmEliminado(25),KmEliminado(29),KmEliminado(27),Nombre Línea,Descripción Motivo Elim,Validacion
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DL219,No se presenta conductor a realizar servicio,0
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,577,No se presenta conductor a realizar servicio,0
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,806,No se presenta conductor a realizar servicio,0


In [621]:
# Numericas
acciones_fms_f['ViajeLineaDesde'] = pd.to_numeric(acciones_fms_f['ViajeLineaDesde'], errors='coerce')
acciones_fms_f['IdViajeDesde'] = pd.to_numeric(acciones_fms_f['IdViajeDesde'], errors='coerce')
acciones_fms_f['ViajeLineaHasta'] = pd.to_numeric(acciones_fms_f['ViajeLineaHasta'], errors='coerce')
acciones_fms_f['IdViajeHasta'] = pd.to_numeric(acciones_fms_f['IdViajeHasta'], errors='coerce')


In [622]:
# Crear nuevas columnas calculadas
acciones_fms_f['ViajeLineaDesde_calc'] = np.where(
    acciones_fms_f['ViajeLineaDesde'].isna(),
    acciones_fms_f['ViajeLineaHasta'] - 1,
    acciones_fms_f['ViajeLineaDesde']
)

acciones_fms_f['IdViajeDesde_calc'] = np.where(
    acciones_fms_f['IdViajeDesde'].isna(),
    acciones_fms_f['IdViajeHasta'] - 1,
    acciones_fms_f['IdViajeDesde']
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(19),KmEliminado(26),KmEliminado(25),KmEliminado(29),KmEliminado(27),Nombre Línea,Descripción Motivo Elim,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,NaN,NaN,DL219,No se presenta conductor a realizar servicio,0,0.0,1.0
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,NaN,NaN,NaN,NaN,577,No se presenta conductor a realizar servicio,0,0.0,1.0
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,NaN,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,NaN,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,NaN,NaN,NaN,NaN,806,No se presenta conductor a realizar servicio,0,0.0,1.0


In [623]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (CreateServiceVo['VEH_SERV_ID'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not CreateServiceVo.loc[filtro].empty:
        # Obtener el primer valor
        tipo = CreateServiceVo.loc[filtro, 'vehServId'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['vehServId'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(26),KmEliminado(25),KmEliminado(29),KmEliminado(27),Nombre Línea,Descripción Motivo Elim,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,NaN,DL219,No se presenta conductor a realizar servicio,0,0.0,1.0,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,NaN,NaN,NaN,577,No se presenta conductor a realizar servicio,0,0.0,1.0,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,NaN,NaN,NaN,806,No se presenta conductor a realizar servicio,0,0.0,1.0,None


In [624]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (CreateServiceVo['VEH_SERV_ID'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not CreateServiceVo.loc[filtro].empty:
        # Obtener el primer valor
        tipo = CreateServiceVo.loc[filtro, 'fromServTripSeq'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['fromServTripSeq'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(25),KmEliminado(29),KmEliminado(27),Nombre Línea,Descripción Motivo Elim,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId,fromServTripSeq
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,NaN,DL219,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,NaN,NaN,577,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,NaN,NaN,806,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None


In [625]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (CreateServiceVo['VEH_SERV_ID'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not CreateServiceVo.loc[filtro].empty:
        # Obtener el primer valor
        tipo = CreateServiceVo.loc[filtro, 'toServTripSeq'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['toServTripSeq'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(29),KmEliminado(27),Nombre Línea,Descripción Motivo Elim,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId,fromServTripSeq,toServTripSeq
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,NaN,DL219,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,NaN,577,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,NaN,806,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None


In [626]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (acciones_change_vehicle['ServBusNuevo'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not acciones_change_vehicle.loc[filtro].empty:
        # Obtener el primer valor
        tipo = acciones_change_vehicle.loc[filtro, 'VEH_SERV_ID'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['ServBusOrg'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,KmEliminado(27),Nombre Línea,Descripción Motivo Elim,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId,fromServTripSeq,toServTripSeq,ServBusOrg
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,DL219,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,577,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,806,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None


In [627]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (acciones_change_vehicle['ServBusNuevo'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not acciones_change_vehicle.loc[filtro].empty:
        # Obtener el primer valor
        tipo = acciones_change_vehicle.loc[filtro, 'ViajeLineaDesdeRef'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['ViajeLineaDesdeRef'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)
acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,Nombre Línea,Descripción Motivo Elim,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId,fromServTripSeq,toServTripSeq,ServBusOrg,ViajeLineaDesdeRef
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,DL219,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,577,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,614,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,806,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None


In [628]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (acciones_change_vehicle['ServBusNuevo'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not acciones_change_vehicle.loc[filtro].empty:
        # Obtener el primer valor
        tipo = acciones_change_vehicle.loc[filtro, 'IdViajeDesdeRef'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['IdViajeDesdeRef'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,Descripción Motivo Elim,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId,fromServTripSeq,toServTripSeq,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,No se presenta conductor a realizar servicio,0,0.0,1.0,None,None,None,None,None,None


In [629]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (acciones_change_vehicle['ServBusNuevo'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not acciones_change_vehicle.loc[filtro].empty:
        # Obtener el primer valor
        tipo = acciones_change_vehicle.loc[filtro, 'ViajeLineaHastaRef'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['ViajeLineaHastaRef'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,Validacion,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId,fromServTripSeq,toServTripSeq,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,0,0.0,1.0,None,None,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,0,0.0,1.0,None,None,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,0,0.0,1.0,None,None,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,0,0.0,1.0,None,None,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,0,0.0,1.0,None,None,None,None,None,None,None


In [630]:
# Llevar servicio original de viaje desglosado al detallado de bus

def calcular_posicion(linea):
    
    filtro = (
        (acciones_change_vehicle['ServBusNuevo'] == linea)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not acciones_change_vehicle.loc[filtro].empty:
        # Obtener el primer valor
        tipo = acciones_change_vehicle.loc[filtro, 'IdViajeHastaRef'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_fms_f['IdViajeHastaRef'] = acciones_fms_f.apply(
    lambda row: calcular_posicion(
        row['VEH_SERV_ID']
    ),
    axis=1
)
acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,ViajeLineaDesde_calc,IdViajeDesde_calc,vehServId,fromServTripSeq,toServTripSeq,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,0.0,1.0,None,None,None,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,0.0,1.0,None,None,None,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,0.0,1.0,None,None,None,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,0.0,1.0,None,None,None,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,0.0,1.0,None,None,None,None,None,None,None,None


In [631]:
# Normalizar posibles vacíos a NaN primero (por si hay '', 'null', etc.)
acciones_fms_f['vehServId'] = acciones_fms_f['vehServId'].replace(['', 'null', 'NULL', 'nan', 'NaN', None], np.nan)
acciones_fms_f['ServBusOrg'] = acciones_fms_f['ServBusOrg'].replace(['', 'null', 'NULL', 'nan', 'NaN', None], np.nan)

# Crear columna ServicioBusReal
acciones_fms_f['ServicioBusReal'] = np.where(
    acciones_fms_f['Validacion'] == 1,
    acciones_fms_f['VEH_SERV_ID'],
    np.where(
        acciones_fms_f['vehServId'].notna(),
        acciones_fms_f['vehServId'],
        acciones_fms_f['ServBusOrg']
    )
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,IdViajeDesde_calc,vehServId,fromServTripSeq,toServTripSeq,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef,ServicioBusReal
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,1.0,NaN,None,None,NaN,None,None,None,None,NaN
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,1.0,NaN,None,None,NaN,None,None,None,None,NaN
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,1.0,NaN,None,None,NaN,None,None,None,None,NaN
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,1.0,NaN,None,None,NaN,None,None,None,None,NaN
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,1.0,NaN,None,None,NaN,None,None,None,None,NaN


In [632]:
# Normalizar vacíos a NaN
cols_vacios = [
    'ViajeLineaDesde','IdViajeDesde',
    'ViajeLineaHasta','IdViajeHasta',
    'ServBusOrg'
]

acciones_fms_f[cols_vacios] = acciones_fms_f[cols_vacios].replace(
    ['', 'null', 'NULL', 'nan', 'NaN', None], np.nan
)

# Crear columnas nuevas vacías
acciones_fms_f['viaje_inicio'] = np.nan
acciones_fms_f['Viaje_inicio_linea'] = np.nan
acciones_fms_f['viaje_fin'] = np.nan
acciones_fms_f['Viaje_fin_linea'] = np.nan


# -----------------------------
# 1. ViajeLineaDesde e IdViajeDesde vacíos
cond1 = acciones_fms_f['ViajeLineaDesde'].isna() & acciones_fms_f['IdViajeDesde'].isna()

acciones_fms_f.loc[cond1, 'viaje_inicio'] = acciones_fms_f.loc[cond1, 'ViajeLineaDesde_calc']
acciones_fms_f.loc[cond1, 'Viaje_inicio_linea'] = acciones_fms_f.loc[cond1, 'IdViajeDesde_calc']


# -----------------------------
# 2. TODOS vacíos
cond2 = (
    acciones_fms_f['ViajeLineaDesde'].isna() &
    acciones_fms_f['IdViajeDesde'].isna() &
    acciones_fms_f['ViajeLineaHasta'].isna() &
    acciones_fms_f['IdViajeHasta'].isna()
)

acciones_fms_f.loc[cond2, 'viaje_inicio'] = acciones_fms_f.loc[cond2, 'fromServTripSeq']
acciones_fms_f.loc[cond2, 'Viaje_inicio_linea'] = acciones_fms_f.loc[cond2, 'fromServTripSeq']

acciones_fms_f.loc[cond2, 'viaje_fin'] = acciones_fms_f.loc[cond2, 'toServTripSeq']
acciones_fms_f.loc[cond2, 'Viaje_fin_linea'] = acciones_fms_f.loc[cond2, 'toServTripSeq']


# -----------------------------
# 3. ViajeLineaHasta e IdViajeHasta vacíos Y ServBusOrg tiene datos
cond3 = (
    acciones_fms_f['ViajeLineaHasta'].isna() &
    acciones_fms_f['IdViajeHasta'].isna() &
    acciones_fms_f['ServBusOrg'].notna()
)

acciones_fms_f.loc[cond3, 'viaje_inicio'] = acciones_fms_f.loc[cond3, 'ViajeLineaDesdeRef']
acciones_fms_f.loc[cond3, 'Viaje_inicio_linea'] = acciones_fms_f.loc[cond3, 'IdViajeDesdeRef']

acciones_fms_f.loc[cond3, 'viaje_fin'] = acciones_fms_f.loc[cond3, 'ViajeLineaHastaRef']
acciones_fms_f.loc[cond3, 'Viaje_fin_linea'] = acciones_fms_f.loc[cond3, 'IdViajeHastaRef']


# -----------------------------
# 4. ViajeLineaHasta e IdViajeHasta vacíos y ServBusOrg NO tiene datos
cond4 = (
    acciones_fms_f['ViajeLineaHasta'].isna() &
    acciones_fms_f['IdViajeHasta'].isna() &
    acciones_fms_f['ServBusOrg'].isna()
)

acciones_fms_f.loc[cond4, 'viaje_inicio'] = acciones_fms_f.loc[cond4, 'ViajeLineaDesde_calc'] + 1
acciones_fms_f.loc[cond4, 'Viaje_inicio_linea'] = acciones_fms_f.loc[cond4, 'IdViajeDesde_calc'] + 1

acciones_fms_f.loc[cond4, 'viaje_fin'] = acciones_fms_f.loc[cond4, 'ViajeLineaDesde_calc'] + 2
acciones_fms_f.loc[cond4, 'Viaje_fin_linea'] = acciones_fms_f.loc[cond4, 'IdViajeDesde_calc'] + 2

# 5. Si NO se cumple ninguna condición anterior
# (conservar valores originales)

cond5 = ~(
    cond2 |   # todos vacíos
    cond3 |   # Hasta vacíos + ServBusOrg con datos
    cond4     # Hasta vacíos + ServBusOrg sin datos
)

acciones_fms_f.loc[cond5, 'viaje_inicio'] = acciones_fms_f.loc[cond5, 'ViajeLineaDesde']
acciones_fms_f.loc[cond5, 'Viaje_inicio_linea'] = acciones_fms_f.loc[cond5, 'IdViajeDesde']

acciones_fms_f.loc[cond5, 'viaje_fin'] = acciones_fms_f.loc[cond5, 'ViajeLineaHasta']
acciones_fms_f.loc[cond5, 'Viaje_fin_linea'] = acciones_fms_f.loc[cond5, 'IdViajeHasta']

acciones_fms_f.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\3868695423.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  acciones_fms_f.loc[cond2, 'viaje_inicio'] = acciones_fms_f.loc[cond2, 'fromServTripSeq']
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\3868695423.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  acciones_fms_f.loc[cond2, 'Viaje_inicio_linea'] = acciones_fms_f.loc[cond2, 'fromServTripSeq']
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\3868695423.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Valu

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef,ServicioBusReal,viaje_inicio,Viaje_inicio_linea,viaje_fin,Viaje_fin_linea
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0


In [633]:
# -----------------------------
# 0. Limpieza más agresiva (la causa de que no entren en cond1 / cond3)
for col in [
    'ViajeLineaDesde','IdViajeDesde',
    'ViajeLineaHasta','IdViajeHasta',
    'ServBusOrg'
]:
    acciones_fms_f[col] = (
        acciones_fms_f[col]
        .astype(str)
        .str.strip()
        .replace(['', 'null', 'NULL', 'nan', 'NaN', 'None', '0'], np.nan)
    )

# Convertir a numérico donde aplique
for col in [
    'ViajeLineaDesde','IdViajeDesde',
    'ViajeLineaHasta','IdViajeHasta'
]:
    acciones_fms_f[col] = pd.to_numeric(acciones_fms_f[col], errors='coerce')


# -----------------------------
# Crear columnas nuevas vacías
acciones_fms_f['viaje_inicio'] = np.nan
acciones_fms_f['Viaje_inicio_linea'] = np.nan
acciones_fms_f['viaje_fin'] = np.nan
acciones_fms_f['Viaje_fin_linea'] = np.nan


# -----------------------------
# 1. ViajeLineaDesde e IdViajeDesde vacíos
cond1 = (
    acciones_fms_f['ViajeLineaDesde'].isna() &
    acciones_fms_f['IdViajeDesde'].isna()
)

acciones_fms_f.loc[cond1, 'viaje_inicio'] = acciones_fms_f.loc[
    cond1, 'ViajeLineaDesde_calc'
]
acciones_fms_f.loc[cond1, 'Viaje_inicio_linea'] = acciones_fms_f.loc[
    cond1, 'IdViajeDesde_calc'
]


# -----------------------------
# 2. TODOS vacíos
cond2 = (
    acciones_fms_f['ViajeLineaDesde'].isna() &
    acciones_fms_f['IdViajeDesde'].isna() &
    acciones_fms_f['ViajeLineaHasta'].isna() &
    acciones_fms_f['IdViajeHasta'].isna()
)

acciones_fms_f.loc[cond2, 'viaje_inicio'] = acciones_fms_f.loc[
    cond2, 'fromServTripSeq'
]
acciones_fms_f.loc[cond2, 'Viaje_inicio_linea'] = acciones_fms_f.loc[
    cond2, 'fromServTripSeq'
]

acciones_fms_f.loc[cond2, 'viaje_fin'] = acciones_fms_f.loc[
    cond2, 'toServTripSeq'
]
acciones_fms_f.loc[cond2, 'Viaje_fin_linea'] = acciones_fms_f.loc[
    cond2, 'toServTripSeq'
]


# -----------------------------
# 3. ViajeLineaHasta e IdViajeHasta vacíos Y ServBusOrg con datos
cond3 = (
    acciones_fms_f['ViajeLineaHasta'].isna() &
    acciones_fms_f['IdViajeHasta'].isna() &
    acciones_fms_f['ServBusOrg'].notna()
)

acciones_fms_f.loc[cond3, 'viaje_inicio'] = acciones_fms_f.loc[
    cond3, 'ViajeLineaDesdeRef'
]
acciones_fms_f.loc[cond3, 'Viaje_inicio_linea'] = acciones_fms_f.loc[
    cond3, 'IdViajeDesdeRef'
]

acciones_fms_f.loc[cond3, 'viaje_fin'] = acciones_fms_f.loc[
    cond3, 'ViajeLineaHastaRef'
]
acciones_fms_f.loc[cond3, 'Viaje_fin_linea'] = acciones_fms_f.loc[
    cond3, 'IdViajeHastaRef'
]


# -----------------------------
# 4. ViajeLineaHasta e IdViajeHasta vacíos y ServBusOrg NO tiene datos
cond4 = (
    acciones_fms_f['ViajeLineaHasta'].isna() &
    acciones_fms_f['IdViajeHasta'].isna() &
    acciones_fms_f['ServBusOrg'].isna()
)

acciones_fms_f.loc[cond4, 'viaje_inicio'] = acciones_fms_f.loc[
    cond4, 'ViajeLineaDesde_calc'
] + 1

acciones_fms_f.loc[cond4, 'Viaje_inicio_linea'] = acciones_fms_f.loc[
    cond4, 'IdViajeDesde_calc'
] + 1

acciones_fms_f.loc[cond4, 'viaje_fin'] = acciones_fms_f.loc[
    cond4, 'ViajeLineaDesde_calc'
] + 2

acciones_fms_f.loc[cond4, 'Viaje_fin_linea'] = acciones_fms_f.loc[
    cond4, 'IdViajeDesde_calc'
] + 2


# -----------------------------
# 5. Si NO se cumple ninguna condición anterior
# (conservar valores originales)
cond5 = ~(cond2 | cond3 | cond4)

acciones_fms_f.loc[cond5, 'viaje_inicio'] = acciones_fms_f.loc[
    cond5, 'ViajeLineaDesde'
]
acciones_fms_f.loc[cond5, 'Viaje_inicio_linea'] = acciones_fms_f.loc[
    cond5, 'IdViajeDesde'
]

acciones_fms_f.loc[cond5, 'viaje_fin'] = acciones_fms_f.loc[
    cond5, 'ViajeLineaHasta'
]
acciones_fms_f.loc[cond5, 'Viaje_fin_linea'] = acciones_fms_f.loc[
    cond5, 'IdViajeHasta'
]


acciones_fms_f.head()


C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\2103225622.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  acciones_fms_f.loc[cond2, 'viaje_inicio'] = acciones_fms_f.loc[
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\2103225622.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  acciones_fms_f.loc[cond2, 'Viaje_inicio_linea'] = acciones_fms_f.loc[
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\2103225622.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None]' has dtype incompatible with float64, pl

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef,ServicioBusReal,viaje_inicio,Viaje_inicio_linea,viaje_fin,Viaje_fin_linea
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0


In [634]:
# 1. Normalizar posibles textos vacíos a NaN
cols = [
    'viaje_inicio', 'Viaje_inicio_linea',
    'viaje_fin', 'Viaje_fin_linea',
    'vehServId'
]

acciones_fms_f[cols] = (
    acciones_fms_f[cols]
    .replace(['', ' ', 'null', 'NULL', 'nan', 'NaN'], np.nan)
)

# 2. Si viaje_inicio y Viaje_inicio_linea están vacíos -> usar datos _calc
cond_inicio_vacio = (
    acciones_fms_f['viaje_inicio'].isna() &
    acciones_fms_f['Viaje_inicio_linea'].isna()
)

acciones_fms_f.loc[cond_inicio_vacio, 'viaje_inicio'] = \
    acciones_fms_f.loc[cond_inicio_vacio, 'ViajeLineaDesde_calc']

acciones_fms_f.loc[cond_inicio_vacio, 'Viaje_inicio_linea'] = \
    acciones_fms_f.loc[cond_inicio_vacio, 'IdViajeDesde_calc']


# 3. Si hay valor en vehServId -> usar fromServTripSeq / toServTripSeq
cond_con_vehserv = acciones_fms_f['vehServId'].notna()

# Sobreescribir INICIO
acciones_fms_f.loc[cond_con_vehserv, 'viaje_inicio'] = \
    acciones_fms_f.loc[cond_con_vehserv, 'fromServTripSeq']

acciones_fms_f.loc[cond_con_vehserv, 'Viaje_inicio_linea'] = \
    acciones_fms_f.loc[cond_con_vehserv, 'fromServTripSeq']


# Sobreescribir FIN
acciones_fms_f.loc[cond_con_vehserv, 'viaje_fin'] = \
    acciones_fms_f.loc[cond_con_vehserv, 'toServTripSeq']

acciones_fms_f.loc[cond_con_vehserv, 'Viaje_fin_linea'] = \
    acciones_fms_f.loc[cond_con_vehserv, 'toServTripSeq']


# 4. Verificación rápida
print("Registros con vehServId:", cond_con_vehserv.sum())
print("Registros con inicio vacío:", cond_inicio_vacio.sum())

acciones_fms_f[['vehServId',
                'viaje_inicio','Viaje_inicio_linea',
                'viaje_fin','Viaje_fin_linea']].head(20)

acciones_fms_f.head()

Registros con vehServId: 1
Registros con inicio vacío: 32


,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef,ServicioBusReal,viaje_inicio,Viaje_inicio_linea,viaje_fin,Viaje_fin_linea
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0


In [635]:
cols = [
    "viaje_inicio", "Viaje_inicio_linea",
    "viaje_fin", "Viaje_fin_linea"
]

acciones_fms_f[cols] = acciones_fms_f[cols].replace(
    ['', ' ', 'nan', 'NaN', 'NULL', 'null', None, np.nan],
    0
)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef,ServicioBusReal,viaje_inicio,Viaje_inicio_linea,viaje_fin,Viaje_fin_linea
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,None,None,None,None,NaN,0.0,1.0,2.0,3.0
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,None,None,None,None,NaN,0.0,1.0,3.0,4.0


In [636]:
acciones_fms_f['viaje_inicio'] = acciones_fms_f['viaje_inicio'].astype(int)
acciones_fms_f['Viaje_inicio_linea'] = acciones_fms_f['Viaje_inicio_linea'].astype(int)
acciones_fms_f['viaje_fin'] = acciones_fms_f['viaje_fin'].astype(int)
acciones_fms_f['Viaje_fin_linea'] = acciones_fms_f['Viaje_fin_linea'].astype(int)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,ServBusOrg,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef,ServicioBusReal,viaje_inicio,Viaje_inicio_linea,viaje_fin,Viaje_fin_linea
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,NaN,None,None,None,None,NaN,0,1,2,3
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,NaN,None,None,None,None,NaN,0,1,2,3
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,NaN,None,None,None,None,NaN,0,1,3,4
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,NaN,None,None,None,None,NaN,0,1,3,4
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,NaN,None,None,None,None,NaN,0,1,3,4


In [637]:

def buscar_instante_iph(servicio, viaje, linea, iph):

    filtro = (
        (iph['Servicio Bus'] == servicio) &
        (iph['Viaje'] == viaje) &
        (iph['Id Línea'] == linea)
    )

    if iph.loc[filtro].empty:
        return None
    else:
        # Tomar la hora máxima (último registro)
        return iph.loc[filtro, 'Instante'].max()


def calcular_hora_instante(acciones_fms_f, iph):

    def obtener_hora(row):

        servicio = row['ServicioBusReal']
        linea    = row['LINE_ID']

        # CASO 1: Si viaje_inicio es 0
        if row['viaje_inicio'] == 0:

            viaje = row['Viaje_inicio_linea']

            # Buscar en iph
            hora_instante_1 = buscar_instante_iph(servicio, viaje, linea, iph)

            return hora_instante_1

        # CASO 2: Si viaje_inicio es diferente de 0
        else:

            # 1. Máximo del viaje_inicio
            viaje_max = row['viaje_inicio']

            # 2. Mínimo de viaje_inicio_linea
            viaje_min = row['Viaje_inicio_linea']

            filtro_max = (
                (iph['Servicio Bus'] == servicio) & 
                (iph['Viaje'] == viaje_max) &
                (iph['Id Línea'] == linea)
            )

            filtro_min = (
                (iph['Servicio Bus'] == servicio) & 
                (iph['Viaje'] == viaje_min) &
                (iph['Id Línea'] == linea)
            )

            if iph.loc[filtro_max].empty or iph.loc[filtro_min].empty:
                return None

            # Hora mayor del viaje máximo
            hora_max = iph.loc[filtro_max, 'Instante'].max()

            # Hora menor del viaje mínimo
            hora_min = iph.loc[filtro_min, 'Instante'].min()

            return max(hora_max, hora_min)

    # Se aplica la función
    acciones_fms_f['hora_instante_1'] = acciones_fms_f.apply(obtener_hora, axis=1)

    return acciones_fms_f


def completar_viajes(acciones_fms_f):

    def llenar_viajes(row):

        # Si hay vehServId
        if pd.notna(row['vehServId']):

            row['viaje_inicio'] = row['fromServTripSeq']
            row['Viaje_inicio_linea'] = row['fromServTripSeq']

            row['viaje_fin'] = row['toServTripSeq']
            row['Viaje_fin_linea'] = row['toServTripSeq']

        # Si están vacíos
        elif pd.isna(row['viaje_inicio']) and pd.isna(row['Viaje_inicio_linea']):

            row['viaje_inicio'] = row['ViajeLineaDesde_calc']
            row['Viaje_inicio_linea'] = row['IdViajeDesde_calc']

        return row

    return acciones_fms_f.apply(llenar_viajes, axis=1)

# Primero completar viajes
acciones_fms_f = completar_viajes(acciones_fms_f)

# Después calcular hora_instante_1
acciones_fms_f = calcular_hora_instante(acciones_fms_f, iph)

# Verificación
acciones_fms_f.head()


,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,ViajeLineaDesdeRef,IdViajeDesdeRef,ViajeLineaHastaRef,IdViajeHastaRef,ServicioBusReal,viaje_inicio,Viaje_inicio_linea,viaje_fin,Viaje_fin_linea,hora_instante_1
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,None,None,None,None,NaN,0,1,2,3,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,None,None,None,None,NaN,0,1,2,3,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,None,None,None,None,NaN,0,1,3,4,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,None,None,None,None,NaN,0,1,3,4,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,None,None,None,None,NaN,0,1,3,4,None


In [638]:
iph_filtrado = iph.copy()
iph_filtrado = iph_filtrado[iph_filtrado['Evento'] != 0]

iph_filtrado.head()

,Fecha,Jornada,TipoDia,Concesionario de Operación,Instante,Servicio Bus,Evento,Id Línea,Tabla,Sublinea,Id Ruta,Id Nodo,Tipo Nodo,Viaje,Servicio Conductor entrante,Turno Entrante,Operador Entrante,Servicio Conductor Saliente,Tipo Vehiculo,Validacion


In [639]:

def buscar_horas(servicio, linea, viaje, iph):
    
    if pd.isna(viaje) or viaje == 0:
        return None, None

    f = (
        (iph_filtrado["Servicio Bus"] == servicio) &
        (iph_filtrado["Id Línea"] == linea) &
        (iph_filtrado["Viaje"] == viaje)
    )

    if iph_filtrado.loc[f].empty:
        return None, None

    return iph_filtrado.loc[f, "Instante"].min(), iph.loc[f, "Instante"].max()


def procesar_todos_los_viajes(acciones_fms_f, iph_filtrado):

    acciones_fms_f["LINE_ID"] = acciones_fms_f["LINE_ID"].astype(str).str.strip()
    acciones_fms_f["ServicioBusReal"] = acciones_fms_f["ServicioBusReal"].astype(str).str.strip()

    iph_filtrado["Id Línea"] = iph_filtrado["Id Línea"].astype(str).str.strip()
    iph_filtrado["Servicio Bus"] = iph_filtrado["Servicio Bus"].astype(str).str.strip()

    # nombres de columnas de viaje
    columnas_viaje = [
        "viaje_inicio",
        "Viaje_inicio_linea",
        "viaje_fin",
        "Viaje_fin_linea"
    ]

    # convertir a numérico
    for col in columnas_viaje:
        acciones_fms_f[col] = pd.to_numeric(acciones_fms_f[col], errors="coerce")

    # procesamos cada fila
    for col in columnas_viaje:

        hora_min_col = f"hora_min_{col}"
        hora_max_col = f"hora_max_{col}"

        acciones_fms_f[[hora_min_col, hora_max_col]] = acciones_fms_f.apply(
            lambda row: buscar_horas(
                row["ServicioBusReal"],
                row["LINE_ID"],
                row[col],
                iph_filtrado
            ),
            axis=1,
            result_type="expand"
        )

    return acciones_fms_f


acciones_fms_f = procesar_todos_los_viajes(acciones_fms_f, iph_filtrado)

acciones_fms_f.head()


,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,Viaje_fin_linea,hora_instante_1,hora_min_viaje_inicio,hora_max_viaje_inicio,hora_min_Viaje_inicio_linea,hora_max_Viaje_inicio_linea,hora_min_viaje_fin,hora_max_viaje_fin,hora_min_Viaje_fin_linea,hora_max_Viaje_fin_linea
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,3,None,None,None,None,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,3,None,None,None,None,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,4,None,None,None,None,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,4,None,None,None,None,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,4,None,None,None,None,None,None,None,None,None


In [640]:
# -------------------------------------------------------
# 2. Función que busca las horas usando SOLO iph_filtrado
# -------------------------------------------------------
def buscar_horas(servicio, linea, viaje, iph_filtrado):
    
    # si el viaje es vacío o 0, no se busca
    if pd.isna(viaje) or viaje == 0:
        return None, None

    # filtro por servicio, línea y viaje
    f = (
        (iph_filtrado["Servicio Bus"] == servicio) &
        (iph_filtrado["Id Línea"] == linea) &
        (iph_filtrado["Viaje"] == viaje)
    )

    # si no existe combinación
    if iph_filtrado.loc[f].empty:
        return None, None

    # *** CORREGIDO ***
    return (
        iph_filtrado.loc[f, "Instante"].min(),
        iph_filtrado.loc[f, "Instante"].max()
    )



# -------------------------------------------------------
# 3. Procesar todos los viajes
# -------------------------------------------------------
def procesar_todos_los_viajes(acciones_fms_f, iph_filtrado):

    # normalizar strings
    acciones_fms_f["LINE_ID"] = acciones_fms_f["LINE_ID"].astype(str).str.strip()
    acciones_fms_f["ServicioBusReal"] = acciones_fms_f["ServicioBusReal"].astype(str).str.strip()
    iph_filtrado["Id Línea"] = iph_filtrado["Id Línea"].astype(str).str.strip()
    iph_filtrado["Servicio Bus"] = iph_filtrado["Servicio Bus"].astype(str).str.strip()

    # columnas de viajes
    columnas_viaje = [
        "viaje_inicio",
        "Viaje_inicio_linea",
        "viaje_fin",
        "Viaje_fin_linea"
    ]

    # convertir valores a numéricos
    for col in columnas_viaje:
        acciones_fms_f[col] = pd.to_numeric(acciones_fms_f[col], errors="coerce")

    # procesar cada fila para cada tipo de viaje
    for col in columnas_viaje:

        hora_min_col = f"hora_min_{col}"
        hora_max_col = f"hora_max_{col}"

        acciones_fms_f[[hora_min_col, hora_max_col]] = acciones_fms_f.apply(
            lambda row: buscar_horas(
                row["ServicioBusReal"],
                row["LINE_ID"],
                row[col],
                iph_filtrado
            ),
            axis=1,
            result_type="expand"
        )

    return acciones_fms_f


# Ejecutar proceso
acciones_fms_f = procesar_todos_los_viajes(acciones_fms_f, iph_filtrado)

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,Viaje_fin_linea,hora_instante_1,hora_min_viaje_inicio,hora_max_viaje_inicio,hora_min_Viaje_inicio_linea,hora_max_Viaje_inicio_linea,hora_min_viaje_fin,hora_max_viaje_fin,hora_min_Viaje_fin_linea,hora_max_Viaje_fin_linea
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,3,None,None,None,None,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,3,None,None,None,None,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,4,None,None,None,None,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,4,None,None,None,None,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,4,None,None,None,None,None,None,None,None,None


Pendiente cruzar datos con las perdidas

In [641]:
# -------------------------------------------
# Función segura para convertir horas
# -------------------------------------------
def safe_to_time(x):
    if pd.isna(x) or x == "" or x is None:
        return None
    try:
        return pd.to_datetime(x, format="%H:%M:%S").time()
    except:
        return None

# -------------------------------------------
# Convertimos horas del dataframe de perdidos
# -------------------------------------------
perdidos["hora_ini"] = perdidos["HORA INICIO"].apply(safe_to_time)

# -------------------------------------------
# Convertimos horas en acciones_fms_f
# -------------------------------------------
cols_hora = [
    "hora_min_viaje_inicio", "hora_max_viaje_inicio",
    "hora_min_Viaje_inicio_linea", "hora_max_Viaje_inicio_linea",
    "hora_min_viaje_fin", "hora_max_viaje_fin",
    "hora_min_Viaje_fin_linea", "hora_max_Viaje_fin_linea"
]

for c in cols_hora:
    acciones_fms_f[c] = acciones_fms_f[c].apply(safe_to_time)

# Columna resultado
acciones_fms_f["MOTIVO_MATCH"] = None

# -------------------------------------------
# Búsqueda por intervalos (ignorando vacíos)
# -------------------------------------------
for idx, row in acciones_fms_f.iterrows():

    servicio = row["ServicioBusReal"]

    # Filtrar perdidos del mismo servicio
    perd_filtro = perdidos[perdidos["SERVICIO VEHICULO"] == servicio]

    if perd_filtro.empty:
        continue

    # Lista de pares de intervalos que revisaremos
    intervalos = [
        ("hora_min_viaje_inicio", "hora_max_viaje_inicio"),
        ("hora_min_Viaje_inicio_linea", "hora_max_Viaje_inicio_linea"),
        ("hora_min_viaje_fin", "hora_max_viaje_fin"),
        ("hora_min_Viaje_fin_linea", "hora_max_Viaje_fin_linea"),
    ]

    motivo = None

    # Revisamos cada posible intervalo
    for h_min, h_max in intervalos:

        min_val = row[h_min]
        max_val = row[h_max]

        # SI EL INTERVALO ESTÁ VACÍO → SE IGNORA
        if min_val is None or max_val is None:
            continue

        # Filtrar si hora_ini cae dentro del intervalo
        mask = (
            (perd_filtro["hora_ini"].notna()) &
            (perd_filtro["hora_ini"] >= min_val) &
            (perd_filtro["hora_ini"] <= max_val)
        )

        coincidencias = perd_filtro[mask]

        if not coincidencias.empty:
            motivo = coincidencias.iloc[0]["MOTIVO"]
            break  # ya encontramos motivo → no seguimos

    # asignar motivo encontrado
    acciones_fms_f.at[idx, "MOTIVO_MATCH"] = motivo

acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,hora_instante_1,hora_min_viaje_inicio,hora_max_viaje_inicio,hora_min_Viaje_inicio_linea,hora_max_Viaje_inicio_linea,hora_min_viaje_fin,hora_max_viaje_fin,hora_min_Viaje_fin_linea,hora_max_Viaje_fin_linea,MOTIVO_MATCH
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,None,None,None,None,None,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,None,None,None,None,None,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,None,None,None,None,None,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,None,None,None,None,None,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,None,None,None,None,None,None,None,None,None,None


In [642]:
# -------------------------------------------
# Función segura para convertir horas
# -------------------------------------------
def safe_to_time(x):
    if pd.isna(x) or x == "" or x is None:
        return None
    try:
        return pd.to_datetime(x, format="%H:%M:%S").time()
    except:
        return None


# -------------------------------------------
# Convertir horas de perdidos
# -------------------------------------------
perdidos["hora_ini"] = perdidos["HORA INICIO"].apply(safe_to_time)


# -------------------------------------------
# Convertir horas de acciones_fms_f
# -------------------------------------------
cols_hora = [
    "hora_min_viaje_inicio", "hora_max_viaje_inicio",
    "hora_min_Viaje_inicio_linea", "hora_max_Viaje_inicio_linea",
    "hora_min_viaje_fin", "hora_max_viaje_fin",
    "hora_min_Viaje_fin_linea", "hora_max_Viaje_fin_linea"
]

for c in cols_hora:
    acciones_fms_f[c] = acciones_fms_f[c].apply(safe_to_time)


# -------------------------------------------
# Reacomodar intervalos invertidos
# -------------------------------------------
def ordenar_intervalo(t1, t2):
    if t1 is None or t2 is None:
        return t1, t2
    return (t1, t2) if t1 <= t2 else (t2, t1)


for idx, row in acciones_fms_f.iterrows():
    for i in range(0, len(cols_hora), 2):
        cmin = cols_hora[i]
        cmax = cols_hora[i+1]
        acciones_fms_f.at[idx, cmin], acciones_fms_f.at[idx, cmax] = \
            ordenar_intervalo(row[cmin], row[cmax])


# -------------------------------------------
# Crear columna resultado
# -------------------------------------------
acciones_fms_f["MOTIVO_MATCH"] = None


# -------------------------------------------
# Realizar el cruce por intervalos y servicio
# -------------------------------------------
for idx, row in acciones_fms_f.iterrows():

    servicio = row["ServicioBusReal"]

    perd_filtro = perdidos[perdidos["SERVICIO VEHICULO"] == servicio]

    if perd_filtro.empty:
        continue

    intervalos = [
        ("hora_min_viaje_inicio", "hora_max_viaje_inicio"),
        ("hora_min_Viaje_inicio_linea", "hora_max_Viaje_inicio_linea"),
        ("hora_min_viaje_fin", "hora_max_viaje_fin"),
        ("hora_min_Viaje_fin_linea", "hora_max_Viaje_fin_linea"),
    ]

    motivo = None

    for h_min, h_max in intervalos:

        min_val = row[h_min]
        max_val = row[h_max]

        if min_val is None or max_val is None:
            continue

        mask = (
            (perd_filtro["hora_ini"].notna()) &
            (perd_filtro["hora_ini"] >= min_val) &
            (perd_filtro["hora_ini"] <= max_val)
        )

        coincidencias = perd_filtro[mask]

        if not coincidencias.empty:
            motivo = coincidencias.iloc[0]["MOTIVO"]
            break

    acciones_fms_f.at[idx, "MOTIVO_MATCH"] = motivo


acciones_fms_f.head()

,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,hora_instante_1,hora_min_viaje_inicio,hora_max_viaje_inicio,hora_min_Viaje_inicio_linea,hora_max_Viaje_inicio_linea,hora_min_viaje_fin,hora_max_viaje_fin,hora_min_Viaje_fin_linea,hora_max_Viaje_fin_linea,MOTIVO_MATCH
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,None,None,None,None,None,None,None,None,None,None
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,None,None,None,None,None,None,None,None,None,None
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,None,None,None,None,None,None,None,None,None,None
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,None,None,None,None,None,None,None,None,None,None
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,None,None,None,None,None,None,None,None,None,None


In [643]:
# ----------------------------
# Parámetros
# ----------------------------
TOLERANCIA_SEC = 10 * 60  # 5 minutos por defecto (ajusta si quieres)

# ----------------------------
# 1) Normalizar vacíos a 0 en las columnas de horas
# ----------------------------
cols_hora = [
    "hora_min_viaje_inicio", "hora_max_viaje_inicio",
    "hora_min_Viaje_inicio_linea", "hora_max_Viaje_inicio_linea",
    "hora_min_viaje_fin", "hora_max_viaje_fin",
    "hora_min_Viaje_fin_linea", "hora_max_Viaje_fin_linea"
]

# Reemplazar múltiples representaciones de vacío por 0 (string 0)
acciones_fms_f = acciones_fms_f.copy()
acciones_fms_f[cols_hora] = acciones_fms_f[cols_hora].replace(
    ['', ' ', 'null', 'NULL', 'nan', 'NaN', None, np.nan],
    0
)

# ----------------------------
# 2) Función para convertir un valor de hora a segundos
#    acepta: int/float (ya segundos), 'HH:MM:SS', pd.Timestamp, pd.Timedelta, datetime.time
# ----------------------------
def hora_a_segundos(h):
    if pd.isna(h):
        return 0
    # si ya es numérico (suponemos segundos)
    if isinstance(h, (int, np.integer, float, np.floating)):
        try:
            return int(h)
        except:
            return 0
    # pandas Timestamp or datetime-like
    try:
        # string 'HH:MM:SS' o 'H:M:S'
        s = str(h)
        # si es '0' o '0.0'
        if s.strip() in ['0', '0.0', '0:00:00', '00:00:00']:
            return 0
        # Try parse as time/datetime
        ts = pd.to_datetime(s, errors='coerce')
        if not pd.isna(ts):
            # si tiene fecha -> extraer la hora del día
            tdelta = pd.Timedelta(hours=ts.hour, minutes=ts.minute, seconds=ts.second)
            return int(tdelta.total_seconds())
    except Exception:
        pass
    # Try if it's a timedelta
    try:
        if isinstance(h, pd.Timedelta):
            return int(h.total_seconds())
    except Exception:
        pass
    # Try if it's time object
    try:
        import datetime
        if isinstance(h, datetime.time):
            return h.hour * 3600 + h.minute * 60 + int(h.second)
    except Exception:
        pass
    # fallback: try splitting as H:M:S
    try:
        parts = str(h).split(':')
        if len(parts) == 3:
            hh = int(parts[0])
            mm = int(parts[1])
            ss = int(float(parts[2]))
            return hh * 3600 + mm * 60 + ss
    except Exception:
        pass
    return 0


# ----------------------------
# 3) Convertir las 8 columnas a segundos (int)
# ----------------------------
for c in cols_hora:
    acciones_fms_f[c + "_sec"] = acciones_fms_f[c].apply(hora_a_segundos).astype(int)

# ----------------------------
# 4) Preparar perdidos: normalizar vacíos y convertir HORA INICIO a segundos
# ----------------------------
perdidos = perdidos.copy()
perdidos["HORA INICIO"] = perdidos["HORA INICIO"].replace(
    ['', ' ', 'null', 'NULL', 'nan', 'NaN', None, np.nan], 0
)
perdidos["HORA_INI_SEC"] = perdidos["HORA INICIO"].apply(hora_a_segundos).astype(int)
perdidos["SERVICIO VEHICULO"] = perdidos["SERVICIO VEHICULO"].astype(str).str.strip()

# también limpiar ServicioBusReal en acciones
acciones_fms_f["ServicioBusReal"] = acciones_fms_f["ServicioBusReal"].astype(str).str.strip()

# ----------------------------
# 5) Match por intervalos (usar columnas _sec)
# ----------------------------
intervalos_sec = [
    ("hora_min_viaje_inicio_sec", "hora_max_viaje_inicio_sec"),
    ("hora_min_Viaje_inicio_linea_sec", "hora_max_Viaje_inicio_linea_sec"),
    ("hora_min_viaje_fin_sec", "hora_max_viaje_fin_sec"),
    ("hora_min_Viaje_fin_linea_sec", "hora_max_Viaje_fin_linea_sec"),
]

acciones_fms_f["MOTIVO_MATCH"] = None
acciones_fms_f["MATCH_DIFF_SEC"] = np.nan

# indexar perdidos por servicio (map servicio -> df subset)
perdidos_by_serv = {s: g for s, g in perdidos.groupby("SERVICIO VEHICULO")}

for idx, row in acciones_fms_f.iterrows():
    serv = row["ServicioBusReal"]
    df_per = perdidos_by_serv.get(serv)
    if df_per is None or df_per.empty:
        continue

    motivo = None
    # recorrer intervalos en segundos
    for cmin, cmax in intervalos_sec:
        vmin = int(row.get(cmin, 0))
        vmax = int(row.get(cmax, 0))
        if vmin == 0 and vmax == 0:
            continue
        # ordenar si están invertidos
        if vmin > vmax:
            vmin, vmax = vmax, vmin
        # buscar coincidencias en df_per por HORA_INI_SEC entre vmin y vmax
        sel = df_per[(df_per["HORA_INI_SEC"] >= vmin) & (df_per["HORA_INI_SEC"] <= vmax)]
        if not sel.empty:
            motivo = sel.iloc[0]["MOTIVO"]
            acciones_fms_f.at[idx, "MOTIVO_MATCH"] = motivo
            acciones_fms_f.at[idx, "MATCH_DIFF_SEC"] = 0
            break

# ----------------------------
# 6) Para los que no emparejaron, buscar la hora más cercana (sin merge_asof)
# ----------------------------
# Preparar arrays por servicio (HORA_INI_SEC y MOTIVO)
perdidos_idx = {}
for serv, g in perdidos.groupby("SERVICIO VEHICULO"):
    times = g["HORA_INI_SEC"].to_numpy(dtype=np.int64)
    motivos = g["MOTIVO"].to_numpy(dtype=object)
    if times.size == 0:
        perdidos_idx[serv] = (np.array([], dtype=np.int64), np.array([], dtype=object))
    else:
        perdidos_idx[serv] = (times, motivos)

# columnas de horas en segundos agregadas
cols_hora_sec = [c + "_sec" for c in cols_hora]

# filas que no tienen motivo aún y que tienen al menos una hora no-cero
mask_no_motivo = acciones_fms_f["MOTIVO_MATCH"].isna()
candidates = acciones_fms_f[mask_no_motivo].copy()

for idx, row in candidates.iterrows():
    serv = row["ServicioBusReal"]
    times_per, motivos_per = perdidos_idx.get(serv, (np.array([], dtype=np.int64), np.array([], dtype=object)))
    if times_per.size == 0:
        continue

    # construir lista de horas disponibles de la fila (no incluir ceros)
    horas_row = []
    for ch in cols_hora_sec:
        val = int(row.get(ch, 0))
        if val != 0:
            horas_row.append(val)
    if len(horas_row) == 0:
        continue

    horas_row = np.array(horas_row, dtype=np.int64)

    # calcular diferencias absolutas entre cada hora_row y todas las times_per (vectorizado)
    # Result: matrix diffs shape (len(horas_row), len(times_per))
    diffs = np.abs(horas_row[:, None] - times_per[None, :])  # broadcasting

    # encontrar la mínima diferencia y su índice (flatten)
    min_idx_flat = np.argmin(diffs)
    i_row, i_per = divmod(min_idx_flat, diffs.shape[1])
    min_diff = int(diffs[i_row, i_per])

    # si la mínima diferencia está dentro de tolerancia, asignar motivo
    if min_diff <= TOLERANCIA_SEC:
        motivo = motivos_per[i_per]
        acciones_fms_f.at[idx, "MOTIVO_MATCH"] = motivo
        acciones_fms_f.at[idx, "MATCH_DIFF_SEC"] = min_diff
    else:
        acciones_fms_f.at[idx, "MATCH_DIFF_SEC"] = min_diff  # registra la mínima diferencia para análisis

# ----------------------------
# 7) Resultados
# ----------------------------
acciones_fms_f["MATCH_DIFF_MIN"] = acciones_fms_f["MATCH_DIFF_SEC"].apply(lambda x: x / 60.0 if pd.notna(x) else np.nan)

print("Matches totales:", acciones_fms_f["MOTIVO_MATCH"].notna().sum())
print("Sin match:", acciones_fms_f["MOTIVO_MATCH"].isna().sum())

acciones_fms_f.head()

Matches totales: 0
Sin match: 320


C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_29816\1629724929.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  acciones_fms_f[cols_hora] = acciones_fms_f[cols_hora].replace(


,APPLY_DATE,ACT_SEQ,OPRTR_ID,LINE_ID,LINE_SERV_ID,VEH_REGISTR_NUM,REGUL_TYPE_ID,PARAM_VALUE,VEH_SERV_ID,REACT_YN,...,hora_min_viaje_inicio_sec,hora_max_viaje_inicio_sec,hora_min_Viaje_inicio_linea_sec,hora_max_Viaje_inicio_linea_sec,hora_min_viaje_fin_sec,hora_max_viaje_fin_sec,hora_min_Viaje_fin_linea_sec,hora_max_Viaje_fin_linea_sec,MATCH_DIFF_SEC,MATCH_DIFF_MIN
4,20260421,556757,105,10690,4,0,5,<= Elimination Vehicle =>\nIdLinea=10690\nServ...,CE1210004,N,...,0,0,0,0,0,0,0,0,NaN,NaN
5,20260421,556903,105,10292,2,0,5,<= Elimination Vehicle =>\nIdLinea=10292\nServ...,CE1780004,N,...,0,0,0,0,0,0,0,0,NaN,NaN
6,20260421,556966,105,10264,29,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0028,N,...,0,0,0,0,0,0,0,0,NaN,NaN
7,20260421,556931,105,10264,13,0,5,<= Elimination Vehicle =>\nIdLinea=10264\nServ...,CE12D0006,N,...,0,0,0,0,0,0,0,0,NaN,NaN
8,20260421,556978,105,10304,4,0,5,<= Elimination Vehicle =>\nIdLinea=10304\nServ...,CE1660003,N,...,0,0,0,0,0,0,0,0,NaN,NaN


In [644]:
print(acciones_fms_f['MOTIVO_MATCH'].count())

0


In [645]:
acciones_fms.to_csv(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/datos_brutos_FMS/Datos_procesados/{dia}_datos_brutos.csv', sep= ';',index=False)

In [646]:
acciones_fms_f.to_csv(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/datos_brutos_FMS/Datos_procesados/{dia}_datos_brutos_filtrados.csv', sep= ';',index=False)

In [647]:
CreateServiceVo.to_csv(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/datos_brutos_FMS/Datos_procesados/{dia}_datos_crear_servicio.csv', sep= ';',index=False)

In [648]:
acciones_change_vehicle.to_csv(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/datos_brutos_FMS/Datos_procesados/{dia}_datos_retomas.csv', sep= ';',index=False)